# 18.1 Sources of nonlinearity

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

We discuss here three ways that nonlinearities come to be included in optimization
models: by dropping a linearity assumption, by constructing a nonlinear function to
achieve a desired effect, and by modeling an inherently nonlinear physical process.

As an example, we describe some nonlinear variants of the linear network flow model
net1.mod introduced in [Chapter 15](../15/15.md) ([Figure 15-2a](../15/15_1_minimumcost_transshipment_models.ipynb#fig-15-2a)). This linear program's objective is
to minimize total shipping cost,

```ampl
minimize Total_Cost:
       sum {(i,j) in LINKS} cost[i,j] * Ship[i,j];
```

where cost[i,j] and Ship[i,j] represent the cost per unit and total units shipped
between cities `i` and j, with LINKS being the set of all city pairs between which shipment
routes exist. The constraints are balance of flow at each city:

```ampl
subject to Balance {k in CITIES}:
       supply[k] + sum {(i,k) in LINKS} Ship[i,k]
       = demand[k] + sum {(k,j) in LINKS} Ship[k,j];
```

with the nonnegative parameters supply[i] and demand[i] representing the units
either available or required at city i.

**Dropping a linearity assumption**

The linear network flow model assumes that each unit shipped from city `i` to city `j`
incurs the same shipping cost, cost[i,j]. [Figure 18-2a](./18_1_sources_of_nonlinearity.ipynb#fig-18-2a) shows a typical plot of shipping
cost versus amount shipped in this case; the plot is a line with slope cost[i,j]
(hence the term linear). The other plots in [Figure 18-2](./18_1_sources_of_nonlinearity.ipynb#fig-18-2) show a variety of other ways,
none of them linear, in which shipping cost could depend on the shipment amount.

In [Figure 18-2b](./18_1_sources_of_nonlinearity.ipynb#fig-18-2b) the cost also tends to increase linearly with the amount shipped, but at
certain critical amounts the cost per unit (that is, the slope of the line) makes an abrupt
change. This kind of function is called piecewise-linear. It is not linear, strictly speaking,
but it is also not smoothly nonlinear. The use of piecewise-linear objectives is the
topic of [Chapter 17](../17/17.md).

In [Figure 18-2c](./18_1_sources_of_nonlinearity.ipynb#fig-18-2c) the function itself jumps abruptly. When nothing is shipped, the shipping
cost is zero; but when there is any shipment at all, the cost is linear starting from a
value greater than zero. In this case there is a fixed cost for using the link from `i` to j,
plus a variable cost per unit shipped. Again, this is not a function that can be handled by
linear programming techniques, but it is also not a smooth nonlinear function. Fixed
costs are most commonly handled by use of integer variables, which are the topic of
[Chapter 20](../20/20.md).

The remaining plots illustrate the sorts of smooth nonlinear functions that we want to
consider in this chapter. [Figure 18-2d](./18_1_sources_of_nonlinearity.ipynb#fig-18-2d) shows a kind of concave cost function. The incremental
cost for each additional unit shipped (that is, the slope of the plot) is great at first,
but becomes less as more units are shipped; after a certain point, the cost is nearly linear.
This is a continuous alternative to the fixed cost function of [Figure 18-2c](./18_1_sources_of_nonlinearity.ipynb#fig-18-2c). It could also
be used to approximate the cost for a situation (resembling [Figure 18-2b](./18_1_sources_of_nonlinearity.ipynb#fig-18-2b)) in which volume
discounts become available as the amount shipped increases.

[Figure 18-2e](./18_1_sources_of_nonlinearity.ipynb#fig-18-2e) shows a kind of convex cost function. The cost is more or less linear for
smaller shipments, but rises steeply as shipment amounts approach some critical amount.
This sort of function would be used to model a situation in which the lowest cost shippers
are used first, while shipping becomes progressively more expensive as the number of
units increases. The critical amount represents, in effect, an upper bound on the shipments.

These are some of the simplest functional forms. The functions that you consider will
depend on the kind of situation that you are trying to model. [Figure 18-2f](./18_1_sources_of_nonlinearity.ipynb#fig-18-2f) shows a possibility
that is neither concave nor convex, combining features of the previous two examples.

Whereas linear functions are essentially all the same except for the choice of coefficients
(or slopes), nonlinear functions can be defined by an infinite variety of different
formulas. Thus in building a nonlinear programming model, it is up to you to derive or
specify nonlinear functions that properly represent the situation at hand. In the objective

![](18_2_nonlinear_cost_functions.png)

<a id='fig-18-2'><center><b>Figure 18-2:</b> Nonlinear cost functions.</center></a>

of the transportation example, for instance, one possibility would be to replace the product
cost[i,j] * Ship[i,j] by

```ampl
(cost1[i,j] + cost2[i,j]*Ship[i,j]) / (1 + Ship[i,j])
       * Ship[i,j]
```

This function grows quickly at small shipment levels but levels off to essentially linear at
larger levels. Thus it represents one way to implement the curve shown in [Figure 18-2d](./18_1_sources_of_nonlinearity.ipynb#fig-18-2d).

Another way to approach the specification of a nonlinear objective function is to
focus on the slopes of the plots in [Figure 18-2](./18_1_sources_of_nonlinearity.ipynb#fig-18-2). In the linear case of [Figure 18-2a](./18_1_sources_of_nonlinearity.ipynb#fig-18-2a), the
slope of the plot is constant; that is why we can use a single parameter `cost[i,j]` to
represent the cost per unit shipped. In the piecewise-linear case of [Figure 18-2b](./18_1_sources_of_nonlinearity.ipynb#fig-18-2b), the
slope is constant within each interval; we can express such piecewise-linear functions as
explained in [Chapter 17](../17/17.md).

In the nonlinear case, however, the slope varies continuously with the amount
shipped. This suggests that we go back to our original linear formulation of the network
flow problem, and turn the parameter `cost[i,j]` into a variable `Cost[i,j]`:

```ampl
var Cost {ORIG,DEST};                   # shipment costs per unit
var Ship {ORIG,DEST} >= 0;              # units to ship

minimize Total_Cost:
       sum {i in ORIG, j in DEST} Cost[i,j] * Ship[i,j];
```

This is no longer a linear objective, because it multiplies a variable by another variable.
We add some equations to specify how the cost relates to the amount shipped:

```ampl
subject to Cost_Relation {i in ORIG, j in DEST}:
       Cost[i,j] =
       (cost1[i,j] + cost2[i,j]*Ship[i,j]) / (1 + Ship[i,j]);
```

These equations are also nonlinear, because they involve division by an expression that
contains a variable. It is easy to see that Cost[i,j] is near cost1[i,j] where shipments
are near zero, but levels off to cost2[i,j]at sufficiently high shipment levels.
Thus the concave cost of [Figure 18-2d](./18_1_sources_of_nonlinearity.ipynb#fig-18-2d) is realized provided that the first cost is greater
than the second.

Assumptions of nonlinearity can be found in constraints as well. The constraints of
the network flow model embody only a weak linearity assumption, to the effect that the
total shipped out of a city is the sum of the shipments to the other cities. But in the production
model of [Figure 1-6a](../01/tut_1_6.ipynb#fig-1-6a), the constraint

```ampl
subject to Time {s in STAGE}:
       sum {p in PROD} (1/rate[p,s]) * Make[p] <= avail[s];
```

embodies a strong assumption that the number of hours used in each stage `s` of making
each product `p` grows linearly with the level of production.
**Achieving a nonlinear effect**

Sometimes nonlinearity arises from a need to model a situation in which a linear function
could not possibly exhibit the desired behavior.

In a network model of traffic flows, as one example, it may be necessary to take congestion
into account. The total time to traverse a shipment link should be essentially a
constant for small shipment amounts, but should increase rapidly towards infinity as the
capacity of the link is approached. No linear function has this property, so we are forced
to make travel time a nonlinear function of shipment load in order to get the desired
effect.

One possibility for expressing the travel time is given by the function

```ampl
time[i,j] + (sens[i,j]*Ship[i,j]) / (1 - Ship[i,j]/cap[i,j])
```

This function is approximately time[i,j] for small values of Ship[i,j], but goes
to infinity as Ship[i,j] approaches cap[i,j]; a third parameter sens[i,j] governs
the shape of the function between the two extremes. This function is always convex,
and so has a graph resembling [Figure 18-2e](./18_1_sources_of_nonlinearity.ipynb#fig-18-2e). ([Exercise 18-4](./18_exercises.ipynb) suggests how this travel time
function can be incorporated into a network model of traffic flows.)

As another example, we may want to allow demand to be satisfied only approximately.
We can model this possibility by introducing a variable Discrepancy[k], to
represent the deviation of the amount delivered from the amount demanded. This variable,
which can be either positive or negative, is added to the right-hand side of the balance
constraint:

```ampl
subject to Balance {k in CITIES}:
       supply[k] + sum {(i,k) in LINKS} Ship[i,k]
       = demand[k] + Discrepancy[k] +
              sum {(k,j) in LINKS} Ship[k,j];
```

One established approach for keeping the discrepancy from becoming too large is to add
a penalty cost to the objective. If this penalty is proportional to the amount of the
discrepancy, then we have a convex piecewise-linear penalty term,

```ampl
minimize Total_Cost:
       sum {(i,j) in LINKS} cost[i,j] * Ship[i,j] +
       sum {k in CITIES} pen * <<-1,1; 0>> Discrepancy[k];
```

where pen is a positive parameter. AMPL readily transforms this objective to a linear
one.

This form of penalty may not achieve the effect that we want, however, because it
penalizes each unit of the discrepancy equally. To discourage large discrepancies, we
would want the penalty to become steadily larger per unit as the discrepancy becomes
worse, but this is not a property that can be achieved by linear penalty functions (or
piecewise-linear ones that have a finite number of pieces). Instead a more appropriate
penalty function would be quadratic:

```ampl
minimize Total_Cost:
       sum {(i,j) in LINKS} cost[i,j] * Ship[i,j] +
       sum {k in CITIES} pen * Discrepancy[k] ˆ 2;
```

Nonlinear objectives that are a sum of squares of some quantities are common in optimization
problems that involve approximation or data fitting.

Modeling an inherently nonlinear process
There are many sources of nonlinearity in models of physical activities like oil refining,
power transmission, and structural design. More often than not, the nonlinearities in
these models cannot be traced to the relaxation of any linearity assumptions, but are a
consequence of inherently nonlinear relationships that govern forces, volumes, currents
and so forth. The forms of the nonlinear functions in physical models may be easier to
determine, because they arise from empirical measurements and the underlying laws of
physics (rather than economics). On the other hand, the nonlinearities in physical models
tend to involve more complicated functional forms and interactions among the variables.

As a simple example, a model of a natural gas pipeline network must incorporate not
only the shipments between cities but also the pressures at individual cities, which are
subject to certain bounds. Thus in addition to the flow variables `Ship[i,j]` the model
must define a variable `Press[k]` to represent the pressure at each city k. If the pressure
is greater at city `i` than at city `j`, then the flow is from `i` to `j` and is related to the pressure
by

```ampl
Flow[i,j]ˆ2 = c[i,j]ˆ2 * (Press[i]ˆ2 - Press[j]ˆ2)
```

where `c[i,j]` is a constant determined by the length, diameter, and efficiency of the
pipe and the properties of the gas. Compressors and valves along the pipeline give rise to
different nonlinear flow relationships. Other kinds of networks, notably for transmission
of electricity, have their own nonlinear flow relationships that are dictated by the physics
of the situation.

If you know the algebraic form of a nonlinear expression that you want to include in
your model, you can probably see a way to write it in AMPL. The next two sections of
this chapter consider some of the specific issues and features relevant to declaring variables
for nonlinear programs and to writing nonlinear expressions. Lest you get carried
away by the ease of writing nonlinear expressions, however, the last section offers some
cautionary advice on solving nonlinear programs.